# NDI sleep-score dating check (read-only)

v1.4 review memo, Section 3 (deliverable 1). Run before tagging Task 3. Runtime > Run all, about 3 minutes.

- CELL 1.0 and CELL 5 are the v1.4 cells, unchanged (CELL 5 provides `load_sleep_score`, the function whose dating rule is checked).
- **SS-1** joins every `sleep_score.csv` row to `sleep.json` by `sleep_log_entry_id = logId` and counts, per participant: rows whose CELL 5 date differs from the matched log's `dateOfSleep`, scores with no matching log, and CELL 5 dates with more than one score. Verdict PASS only if the mismatches are among the 3 known shifted scores (p03 1, p16 2).

Writes only to `results_diagnostics/` (sleep_score_check_summary.csv, sleep_score_check_rows.csv, sleep_score_check_verdict.json).

In [1]:
# ================================================================
# CELL 1.0  CONFIGURATION  (edit paths to match your Drive)
# Brief Section 6: all parameters in one place, fixed seed
# ================================================================

#  Google Drive paths
PIPELINE_MODULE   = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
DEPRESJON_DIR     = "/content/drive/MyDrive/ndi_project/depresjon-dataset/data"
PMDATA_DIR        = "/content/drive/MyDrive/ndi_project/pmdata"
PMDATA_OVERVIEW   = "/content/drive/MyDrive/ndi_project/pmdata/participant-overview.xlsx"
RESULTS_DIR       = "/content/drive/MyDrive/ndi_project/results"

#  Reproducibility seed (Brief Section 6)
RANDOM_SEED = 42

#  Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add pipeline module to path
sys.path.insert(0, os.path.dirname(PIPELINE_MODULE))

print("Drive mounted.")
print("DEPRESJON dir exists:", os.path.isdir(DEPRESJON_DIR))
print("PMData    dir exists:", os.path.isdir(PMDATA_DIR))
print("Pipeline  exists    :", os.path.isfile(PIPELINE_MODULE))

Mounted at /content/drive
Drive mounted.
DEPRESJON dir exists: True
PMData    dir exists: True
Pipeline  exists    : True


In [2]:
# ================================================================
# CELL 5 -- LOAD PMDATA DATASET   [v1.4: two changes, marked 'v1.4' below]
# Actual structure:
#   pmdata/p01/fitbit/heart_rate.json
#   pmdata/p01/fitbit/resting_heart_rate.json
#   pmdata/p01/fitbit/sleep.json
#   pmdata/p01/fitbit/sleep_score.csv
#   pmdata/p01/fitbit/steps.json
#   pmdata/p01/fitbit/lightly_active_minutes.json
#   pmdata/p01/fitbit/moderately_active_minutes.json
#   pmdata/p01/fitbit/very_active_minutes.json
#   pmdata/p01/fitbit/sedentary_minutes.json
#   pmdata/p01/pmsys/wellness.csv    <- daily mood/wellness self-report
#   pmdata/p01/pmsys/srpe.csv
#   pmdata/participant-overview.xlsx <- max HR, age, gender per participant
# ================================================================
import os, glob, json as _json
import pandas as pd
import numpy as np
import openpyxl  # for reading xlsx

#  Load participant overview
try:
    overview = pd.read_excel(PMDATA_OVERVIEW, header=1)
    overview.columns = [str(c).strip().lower().replace(" ","_")
                        for c in overview.columns]
    overview = overview.dropna(subset=[overview.columns[0]])
    print("participant-overview columns:", list(overview.columns))
    print(overview.head(3))
except Exception as e:
    print(f"Could not load participant-overview: {e}")
    overview = pd.DataFrame()

#  Helper: load one JSON Fitbit file
def load_fitbit_json(path):
    """Load a Fitbit JSON file. Returns list of {dateTime, value} dicts."""
    try:
        with open(path) as f:
            data = _json.load(f)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

def fitbit_to_daily(entries, value_key="value", agg="mean"):
    """Aggregate minute/entry-level Fitbit data to daily values."""
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get(value_key, entry.get("value", None))
        if date and val is not None:
            try:
                val = float(val)
                rows.setdefault(date, []).append(val)
            except (ValueError, TypeError):
                pass
    if agg == "mean":
        return {d: np.mean(v) for d,v in rows.items()}
    elif agg == "sum":
        return {d: np.sum(v) for d,v in rows.items()}
    elif agg == "max":
        return {d: np.max(v) for d,v in rows.items()}
    return rows

#  Helper: load resting heart rate JSON
def load_resting_hr(path):
    """resting_heart_rate.json has nested value.value structure."""
    entries = load_fitbit_json(path)
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get("value",{})
        if isinstance(val, dict):
            rhr = val.get("value", val.get("restingHeartRate", None))
        else:
            rhr = val
        if date and rhr is not None:
            try:
                rows[date] = float(rhr)
            except (ValueError, TypeError):
                pass
    return rows

#  Helper: load sleep.json
def load_sleep(path):
    """Extract sleep efficiency and minutes asleep from sleep.json."""
    entries = load_fitbit_json(path)
    eff_rows = {}; min_rows = {}
    for entry in entries:
        # Each entry is one sleep session
        date = str(entry.get("dateOfSleep", entry.get("dateTime","")))[:10]
        eff  = entry.get("efficiency", None)
        mins = entry.get("minutesAsleep", None)
        if date and eff is not None:
            try:
                eff_rows[date] = float(eff)
            except (ValueError, TypeError):
                pass
        if date and mins is not None:
            try:
                min_rows[date] = float(mins)
            except (ValueError, TypeError):
                pass
    return eff_rows, min_rows

#  Helper: load sleep_score.csv
def load_sleep_score(path):
    """sleep_score.csv: timestamp, overall_score, composition_score, ..."""
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        # Timestamp column
        ts_col = next((c for c in df.columns if "timestamp" in c or "date" in c), None)
        score_col = next((c for c in df.columns if "overall" in c or "score" in c), None)
        if ts_col and score_col:
            df["date"] = pd.to_datetime(df[ts_col]).dt.strftime("%Y-%m-%d")
            return dict(zip(df["date"], pd.to_numeric(df[score_col], errors="coerce")))
    except Exception:
        pass
    return {}

#  Helper: load wellness.csv
def load_wellness(path):
    """
    pmsys/wellness.csv: daily self-report.
    Columns typically: date, fatigue, mood, readiness, sleep_duration,
                       sleep_quality, stress, soreness
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        if "date" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        return df
    except Exception:
        return pd.DataFrame()

#  Helper: load googledocs/reporting.csv
def load_reporting(path):
    """
    reporting.csv: date, timestamp, meals, weight, glasses_of_fluid, alcohol
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        date_col = next((c for c in df.columns if "date" in c), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
            return df
    except Exception:
        pass
    return pd.DataFrame()

#  Main loader
def load_pmdata_participant(pid_dir):
    """
    Load all data for one PMData participant.
    Returns a daily DataFrame with all available signals.
    """
    fitbit_dir    = os.path.join(pid_dir, "fitbit")
    pmsys_dir     = os.path.join(pid_dir, "pmsys")
    googledocs_dir= os.path.join(pid_dir, "googledocs")

    daily = {}  # date -> feature dict

    def add_series(series_dict, col_name):
        for date, val in series_dict.items():
            daily.setdefault(date, {})[col_name] = val

    # Resting heart rate (most reliable HR signal for daily level)
    rhr_path = os.path.join(fitbit_dir, "resting_heart_rate.json")
    if os.path.isfile(rhr_path):
        add_series(load_resting_hr(rhr_path), "resting_hr")

    # Steps
    steps_path = os.path.join(fitbit_dir, "steps.json")
    if os.path.isfile(steps_path):
        add_series(fitbit_to_daily(load_fitbit_json(steps_path), agg="sum"), "steps")

    # Activity minutes
    for fname, col in [
        ("lightly_active_minutes.json",    "lightly_active_min"),
        ("moderately_active_minutes.json", "moderately_active_min"),
        ("very_active_minutes.json",       "very_active_min"),
        ("sedentary_minutes.json",         "sedentary_min"),
    ]:
        p = os.path.join(fitbit_dir, fname)
        if os.path.isfile(p):
            add_series(fitbit_to_daily(load_fitbit_json(p), agg="sum"), col)

    # Sleep
    sleep_path = os.path.join(fitbit_dir, "sleep.json")
    if os.path.isfile(sleep_path):
        eff, mins = load_sleep(sleep_path)
        add_series(eff,  "sleep_efficiency")
        add_series(mins, "sleep_minutes")

    # Sleep score
    ss_path = os.path.join(fitbit_dir, "sleep_score.csv")
    if os.path.isfile(ss_path):
        add_series(load_sleep_score(ss_path), "sleep_score")

    if not daily:
        return pd.DataFrame()

    # Build base daily DataFrame
    df = pd.DataFrame.from_dict(daily, orient="index")
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Active minutes total
    act_cols = [c for c in ["lightly_active_min","moderately_active_min","very_active_min"] if c in df.columns]
    if act_cols:
        df["active_minutes_total"] = df[act_cols].sum(axis=1)

    # v1.4: wellness is NOT merged into days_df (v1.4 memo, Section 4). It is built as a separate
    # table under rules W1-W4 in cell W-1 and joined only when the Task 3 target is built.

    # Merge reporting (weight, fluid)
    r_path = os.path.join(googledocs_dir, "reporting.csv")
    if os.path.isfile(r_path):
        rep = load_reporting(r_path)
        if not rep.empty:
            rep["date"] = pd.to_datetime(rep["date"])
            rep_agg = rep.groupby("date").agg(
                weight         = ("weight", "mean"),
                glasses_of_fluid=("glasses_of_fluid","mean") if "glasses_of_fluid" in rep.columns else ("date","count"),
            ).reset_index()
            df = df.merge(rep_agg, on="date", how="left")

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


#  Load all participants
pmdata_participants = []
participant_dirs = sorted([
    os.path.join(PMDATA_DIR, d)
    for d in os.listdir(PMDATA_DIR)
    if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")
])
print(f"Found {len(participant_dirs)} participant directories")

for pdir in participant_dirs:
    pid = os.path.basename(pdir)
    try:
        daily = load_pmdata_participant(pdir)
        if len(daily) < 7:
            print(f"  SKIP {pid}: only {len(daily)} days")
            continue

        # Get max HR from overview
        max_hr = np.nan
        if not overview.empty:
            hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
            id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
            if hr_col and id_col:
                row = overview[overview[id_col].astype(str).str.lower().str.strip() == pid.lower()]
                if not row.empty:
                    max_hr = pd.to_numeric(row.iloc[0][hr_col], errors="coerce")   # v1.4: p12's cell is ' ' (v1.4 memo, Sec. 2); not a model feature

        pmdata_participants.append(dict(
            pid       = pid,
            group     = "pmdata",
            is_depressed = 0,
            max_hr    = max_hr,
            n_days    = len(daily),
            days_df   = daily,
        ))
        cols_found = [c for c in daily.columns if c != "date"]
        print(f"  OK {pid}: {len(daily)} days  max_HR={max_hr:.0f}  signals={cols_found[:6]}")
    except Exception as e:
        print(f"  ERROR {pid}: {e}")

print(f"\nPMData: {len(pmdata_participants)} participants loaded")
if pmdata_participants:
    print(f"  Median days: {np.median([p['n_days'] for p in pmdata_participants]):.0f}")
    all_cols = set()
    for p in pmdata_participants:
        all_cols.update(p['days_df'].columns.tolist())
    print(f"  All available columns: {sorted(all_cols)}")

participant-overview columns: ['participant_id', 'age', 'height', 'gender', 'a_or_b_person', 'max_heart_rate', 'date', 'minutes', 'seconds', 'stride_walk', 'stride_run']
  participant_id  age  height gender a_or_b_person max_heart_rate  \
0            p01   48     195   male             A            182   
1            p02   60     180   male             A            169   
2            p03   25     184   male             A            157   

                  date minutes seconds  stride_walk  stride_run  
0  2019-11-26 00:00:00      29      33         80.9       102.9  
1  2019-12-15 00:00:00      23      51         74.7        92.4  
2  2019-12-30 00:00:00      33      22          NaN         NaN  
Found 16 participant directories
  OK p01: 152 days  max_HR=182  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']


/tmp/ipykernel_1184/789389307.py:150: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")


  OK p02: 152 days  max_HR=169  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']
  OK p03: 152 days  max_HR=157  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']
  OK p04: 152 days  max_HR=195  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']


/tmp/ipykernel_1184/789389307.py:150: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")


  OK p05: 152 days  max_HR=184  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']


...

/tmp/ipykernel_1184/789389307.py:150: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")


  OK p13: 152 days  max_HR=186  signals=['steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min', 'sleep_efficiency']


/tmp/ipykernel_1184/789389307.py:150: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")


  OK p14: 141 days  max_HR=190  signals=['resting_hr', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min', 'steps']
  OK p15: 146 days  max_HR=nan  signals=['resting_hr', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min', 'steps']
  OK p16: 152 days  max_HR=199  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']

PMData: 16 participants loaded
  Median days: 152
  All available columns: ['active_minutes_total', 'date', 'glasses_of_fluid', 'lightly_active_min', 'moderately_active_min', 'resting_hr', 'sedentary_min', 'sleep_efficiency', 'sleep_minutes', 'sleep_score', 'steps', 'very_active_min', 'weight']


/tmp/ipykernel_1184/789389307.py:150: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")


In [3]:
# SS-1  Sleep-score dating check (v1.4 review memo, Section 3). Read-only; writes results_diagnostics/ only.
# Each sleep_score.csv row is joined to sleep.json by sleep_log_entry_id = logId. For every participant it counts
# rows where the date CELL 5 assigns differs from the matched log's dateOfSleep, scores with no matching log, and
# CELL 5 dates with more than one score. The date rule is CELL 5's own (checked against load_sleep_score).
import os, json
import numpy as np, pandas as pd
_need = ["PMDATA_DIR", "RESULTS_DIR", "load_sleep_score"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run CELL 1.0 and CELL 5 first. Missing: {_miss}"
DIAG_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "results_diagnostics"); os.makedirs(DIAG_DIR, exist_ok=True)
KNOWN_SHIFTED = {"p03": 1, "p16": 2}          # S-0 (v1.4 pre-checks): local date differs from the CELL 5 date
_tz_re = r"(?:Z|[+-]\d\d:?\d\d)$"
summary, detail = [], []
for pid in sorted(d for d in os.listdir(PMDATA_DIR) if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")):
    ss_path = os.path.join(PMDATA_DIR, pid, "fitbit", "sleep_score.csv")
    sj_path = os.path.join(PMDATA_DIR, pid, "fitbit", "sleep.json")
    if not os.path.isfile(ss_path):
        summary.append(dict(pid=pid, sleep_scores=0, note="no sleep_score.csv")); continue
    s = pd.read_csv(ss_path); s.columns = [c.strip().lower() for c in s.columns]          # as CELL 5
    ts_col = next((c for c in s.columns if "timestamp" in c or "date" in c), None)         # as CELL 5
    sc_col = next((c for c in s.columns if "overall" in c or "score" in c), None)          # as CELL 5
    cell5_date = pd.to_datetime(s[ts_col]).dt.strftime("%Y-%m-%d")                         # CELL 5 rule
    ref = load_sleep_score(ss_path)                                                        # CELL 5 function itself
    assert set(ref) == set(cell5_date.dropna()), f"{pid}: re-implemented CELL 5 date rule differs from load_sleep_score"
    raw = s[ts_col].astype(str).str.strip()
    local_date = (pd.to_datetime(raw, errors="coerce", utc=True).dt.tz_convert("Europe/Oslo").dt.strftime("%Y-%m-%d")
                  .where(raw.str.contains(_tz_re, regex=True), cell5_date))
    has_id = "sleep_log_entry_id" in s.columns
    logs = json.load(open(sj_path)) if os.path.isfile(sj_path) else []
    logs = logs if isinstance(logs, list) else []
    log_has_id = any("logId" in e for e in logs)
    by_id = {}
    for e in logs:
        if "logId" in e:
            by_id.setdefault(str(e["logId"]), []).append(e)
    dup_logids = sum(len(v) > 1 for v in by_id.values())
    ids = s["sleep_log_entry_id"].map(lambda v: str(int(v)) if pd.notna(v) else None) if has_id else pd.Series([None] * len(s))
    dos = ids.map(lambda i: str(by_id[i][0].get("dateOfSleep", ""))[:10] if i in by_id else None)
    matched = dos.notna()
    mismatch = matched & (cell5_date != dos)
    shifted = local_date != cell5_date
    multi_dates = cell5_date.value_counts()
    multi_dates = set(multi_dates[multi_dates > 1].index)
    for i in s.index:
        if mismatch[i] or not matched[i] or shifted[i] or cell5_date[i] in multi_dates:
            detail.append(dict(pid=pid, sleep_log_entry_id=ids[i], timestamp=raw[i], cell5_date=cell5_date[i],
                               local_date=local_date[i], matched_log=bool(matched[i]), dateOfSleep=dos[i],
                               mismatch=bool(mismatch[i]), local_differs_from_cell5=bool(shifted[i]),
                               cell5_date_has_2plus_scores=cell5_date[i] in multi_dates, score=s[sc_col][i]))
    summary.append(dict(pid=pid, sleep_scores=len(s), id_column_in_sleep_score=has_id, logId_in_sleep_json=log_has_id,
                        sleep_json_logs=len(logs), duplicate_logIds=dup_logids, matched=int(matched.sum()),
                        no_matching_log=int((~matched).sum()), cell5_date_differs_from_dateOfSleep=int(mismatch.sum()),
                        of_which_local_date_equals_dateOfSleep=int((mismatch & (local_date == dos)).sum()),
                        local_date_differs_from_cell5_date=int(shifted.sum()),
                        cell5_dates_with_2plus_scores=len(multi_dates)))
summ = pd.DataFrame(summary); det = pd.DataFrame(detail)
summ.to_csv(os.path.join(DIAG_DIR, "sleep_score_check_summary.csv"), index=False)
det.to_csv(os.path.join(DIAG_DIR, "sleep_score_check_rows.csv"), index=False)
pd.set_option("display.width", 250)
print(summ.to_string(index=False))
print("\nRows that are mismatched, unmatched, shifted or on a date with 2+ scores:")
print(det.to_string(index=False) if len(det) else "none")

_have = summ[summ["sleep_scores"] > 0]
id_ok = bool(_have["id_column_in_sleep_score"].all() and _have["logId_in_sleep_json"].all())
mm = det[det["mismatch"]] if len(det) else det
mm_unknown = mm[~mm["local_differs_from_cell5"]] if len(mm) else mm
mm_known = mm[mm["local_differs_from_cell5"]] if len(mm) else mm
known_ok = all(int(((mm_known["pid"] == p)).sum()) <= k for p, k in KNOWN_SHIFTED.items()) if len(mm_known) else True
extra_known = sorted(set(mm_known["pid"]) - set(KNOWN_SHIFTED)) if len(mm_known) else []
if not id_ok:
    verdict, why = "STOP", "sleep_log_entry_id or logId is missing for some participants: tell ma'am (memo, Section 3)"
elif len(mm_unknown) or not known_ok or extra_known:
    verdict, why = "STOP", (f"{len(mm)} mismatches, not only the known shifted scores (p03 1, p16 2): send the counts "
                            "to ma'am before tagging")
else:
    verdict, why = "PASS", (f"the only mismatches are among the known shifted scores ({len(mm)} rows: "
                            + (", ".join(f"{r.pid} {r.cell5_date}->{r.dateOfSleep}" for r in mm.itertuples()) or "none")
                            + "); record as a deviation and keep the v1.4 features unchanged for Task 3")
out = dict(verdict=verdict, reason=why, mismatches_total=int(len(mm)), mismatches_known_shifted=int(len(mm_known)),
           mismatches_other=int(len(mm_unknown)), scores_without_matching_log=int(_have["no_matching_log"].sum()),
           cell5_dates_with_2plus_scores=int(_have["cell5_dates_with_2plus_scores"].sum()),
           sleep_scores_total=int(_have["sleep_scores"].sum()))
with open(os.path.join(DIAG_DIR, "sleep_score_check_verdict.json"), "w") as f:
    json.dump(out, f, indent=2)
print("\n" + json.dumps(out, indent=2))
print(f"\nSLEEP-SCORE CHECK: {verdict} -- {why}")

pid  sleep_scores  id_column_in_sleep_score  logId_in_sleep_json  sleep_json_logs  duplicate_logIds  matched  no_matching_log  cell5_date_differs_from_dateOfSleep  of_which_local_date_equals_dateOfSleep  local_date_differs_from_cell5_date  cell5_dates_with_2plus_scores
p01           150                      True                 True              155                 5      149                1                                    0                                       0                                   0                              0
p02           138                      True                 True              158                 5      138                0                                    0                                       0                                   0                              0
p03            74                      True                 True               84                 3       74                0                                    0                            